<!-- nav -->
[← 14 · One constraint, many languages](14_One_Constraint_Many_Languages.ipynb) · [Home](../../README.md) · [Equivalence of the implementations →](../../docs/EQUIVALENCE.md)

# 15 · Constraints for firmware and hardware

**The problem.** The logger's firmware, in C, checks each packet before sending it, and its FPGA's verification
engineers check the same condition with SystemVerilog assertions. In C, `uint8_t + uint8_t` is an `int`, and signed
overflow is undefined. In SystemVerilog, a bit can be `x`, unknown, or `z`, undriven. A constraint that is right in
TypeScript can be wrong in either.

The C and SystemVerilog dialects model those languages' rules exactly, so you can translate a packet's constraint,
written once in Patterns, and evaluate it the way the firmware and the hardware will. A Patterns term carries no width: its
ints are unbounded, and what C's and SystemVerilog's types make of a value is theirs. Giving Patterns' terms domains,
a 16-bit integer or ISO/IEC 11404's `modulo(256)`, waits for domain inference
([CAP-232: domains inferred for terms](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-232.md)).

In [1]:
import { Simplifiers, Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import * as C from "@mbse/expressions/C/Expressions";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as Translators from "@mbse/expressions/Translators";
import { cold_chain } from "./toolkit.js";

const store = cold_chain(); // with C's and SystemVerilog's kinds installed, and their types
const [Expression, Term] = [store.Expressions.Framework.Expression, store.Expressions.Framework.Term];
const [r, s] = [E.variable("raw"), E.variable("status")];
const packet_ok = store.transaction(() => Expression((e: any) => e.root(r.ge(200n).and_(r.le(800n)).and_(s.bitand(0b100n).eq(0n)))));
const to_c = Translators.between(E.DIALECT, C.Expressions.DIALECT);
const to_sv = Translators.between(E.DIALECT, SystemVerilog.Expressions.DIALECT);

<a id="T15.1"></a>
## 15.1 · C

A translation is a writer of the target's term, written in a transaction like any other. C's standard leaves most of its
types to the implementation, and an ABI fixes them, each ABI a dialect of its own holding those types as data, as
[tutorial 2](02_One_Value_A_Domain_In_Each_Language.ipynb#T02.3) shows. `cold_chain` installs and defines two: x86-64's System V ABI (`SystemV.AMD64`) and
Arm's AAPCS64 (`Arm.AAPCS64`). The expression holding a C
term names the ABI its terms are read under, and C's rules read its types through it:

In [2]:
const amd64 = store.Instances.Expressions.SystemV.AMD64; // x86-64's ABI, by its global name
const c = store.transaction(() => Expression((e: any) => e.root(to_c.forward(packet_ok)).abi([amd64])));
console.log(C.Text.ToText(c));
for (const status of [0n, 0b100n]) {
  console.log(String(status), C.Simplifiers.OfAny(c, { raw: 523n, status }).evaluate());
}

raw >= 200 && raw <= 800 && (status & 4) == 0


0 true


4 false


<a id="T15.2"></a>
## 15.2 · Where C differs

C's integers have widths: an `int` is 32 bits. Patterns' are unbounded, so a sum Patterns computes exactly can overflow in
C, and signed overflow is undefined behavior in C: a compiler may assume it never happens. The rules throw instead of picking a result, which is what a test should do:

In [3]:
const increment = store.transaction(() => Term(r.add(1n)));
const increment_c = store.transaction(() => Expression((e: any) => e.root(to_c.forward(increment)).abi([amd64])));
console.log(C.Text.ToText(increment_c), "in Patterns:", String(Simplifiers.OfAny(increment, { raw: 2147483647n }).value));
try {
  C.Simplifiers.OfAny(increment_c, { raw: 2147483647n }).evaluate();
} catch (error) {
  console.log("C:", (error as Error).message);
}

raw + 1 in Patterns: 2147483648


C: signed overflow in +: 2147483648 does not fit 'int'


C's own types are C's: a `uint8_t` is promoted to `int` before it is added, so the sum is 300, not 44. Patterns has no
counterpart for a cast to a type of C's, or a literal of a type its suffix gives it (`5u`), yet, since its domain waits for
domain inference
([CAP-232: domains inferred for terms](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-232.md)), and the translator
refuses it, naming it:

In [4]:
const CE = C.Expressions;
const uint8_t = (t: any) => t.typeid((i: any) => i.specifiers([ // the type name `uint8_t`, written through the builders
  (s: any) => s.namedtypespecifier((n: any) => n.name((m: any) => m.identifier((d: any) => d.spelling("uint8_t"))))]));
const total = store.transaction(() => Expression(
  (e: any) => e.root(CE.binary("+", CE.cast(uint8_t, 200n), CE.cast(uint8_t, 100n))).abi([amd64])));
console.log(C.Text.ToText(total), "=", String(C.Simplifiers.OfAny(total, {}).evaluate()));
try {
  to_c.backward(total);
} catch (error) {
  console.log((error as Error).message);
}

(uint8_t)200 + (uint8_t)100 = 300


C castexpression has no Patterns counterpart


Where the ABIs differ, the same term folds as each says. A plain `char` is signed on x86-64 and unsigned on Arm's
AArch64, so 200 narrowed to a `char` is -56 on the one and 200 on the other. C writes no literal of a `char`, so the
cast stays as it is; compared with 0, it folds, to a `bool`:

In [5]:
const char = (t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.primitivetypespecifier((p: any) => p.keyword("char"))]));
for (const [name, abi] of [["x86-64", amd64], ["AArch64", store.Instances.Expressions.Arm.AAPCS64]] as [string, unknown][]) {
  const narrowed = store.transaction(() => Expression((e: any) => e.root(CE.binary("<", CE.cast(char, 200n), 0n)).abi([abi])));
  console.log(name, C.Text.ToText(narrowed), "=", String(C.Simplifiers.OfAny(narrowed, {}).evaluate()));
}

x86-64 (char)200 = -56


AArch64 (char)200 = 200


<a id="T15.3"></a>
## 15.3 · SystemVerilog

SystemVerilog's integers are 4-state vectors. Its types are the domains its dialect stores, as
[tutorial 2](02_One_Value_A_Domain_In_Each_Language.ipynb#T02.4) shows; `cold_chain` writes them (`SystemVerilog.define`), and the rules read them through the expression holding the term, so the translation is written as an `Expression`. Its rules give a 4-state value, written as
SystemVerilog would display it:

In [6]:
const sv = store.transaction(() => Expression((e: any) => e.root(to_sv.forward(packet_ok))));
console.log(SystemVerilog.Text.ToText(sv));
console.log(String(SystemVerilog.Simplifiers.OfAny(sv, { raw: 523n, status: 0n }).evaluate()));

raw >= 200 && raw <= 800 && (status & 4) == 0


1'b1


<a id="T15.4"></a>
## 15.4 · `x` and `z`

In simulation, a register that was never reset holds `x`. Whether that matters depends on the bit: the mask
looks only at bit 2, so an `x` elsewhere doesn't change the answer, but an `x` in bit 2 makes the whole condition `x`,
which an assertion reports as a failure to know. `Logic.of` makes a 4-state value from its bits:

In [7]:
const Logic = SystemVerilog.Domains.Logic;
for (const bits of ["00000000", "00000100", "000000x0", "00000x00"]) {
  console.log(bits, String(SystemVerilog.Simplifiers.OfAny(sv, { raw: 523n, status: Logic.of(bits) }).evaluate()));
}

00000000 1'b1


00000100 1'b0


000000x0 1'b1


00000x00 1'bx


This is Kleene's unknown again, in hardware's terms: `x` where the data doesn't decide.

<a id="T15.5"></a>
## 15.5 · What has no counterpart

`has` asks whether a property is present. A C struct always has its members, so there's nothing in C to translate
it to yet: its translation to C++ is planned
([CAP-235: `has` translated to C++](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-235.md)), and to C, blocked
([CAP-236: `has` translated to C](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-236.md)). SystemVerilog's counterpart is a known gap, still to close
([CAP-103: SystemVerilog's other operations translated](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-103.md)). Both translators refuse, naming the term:

In [8]:
const reported = store.transaction(() => Term(E.variable("shipment").has("celsius")));
for (const translator of [to_c, to_sv]) {
  try {
    translator.forward(reported);
  } catch (error) {
    console.log((error as Error).message);
  }
}

Patterns call 'has' has no C counterpart


Patterns call 'has' has no SystemVerilog counterpart


## Why it works this way

- **Model the language, don't approximate it.** Firmware and hardware bugs live in exactly the corners where languages
  differ: promotions, wrapping, undefined behavior, `x`. A simplifier that follows C's or IEEE 1800's rules finds them
  in a test, before the hardware does
  ([CAP-100: six languages' expression dialects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-100.md#dialects)).
- **The neutral constraint stays neutral.** Patterns doesn't take C's promotions or SystemVerilog's sizing. The translator
  writes them out, and the target's rules apply them
  ([CAP-101: translation between dialects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-101.md#translators)).
- **Refuse rather than approximate.** A translator that met a term it has no counterpart for refuses, naming it, since a
  silent approximation would be a bug in the translation
  ([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)). Some refusals are gaps still to close, such as `has` in SystemVerilog
  ([CAP-103: SystemVerilog's other operations translated](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-103.md)).

That's the series: a constraint written once, as data, checked, stored, analyzed, specialized, and checked in each
language its users work in.

<!-- nav -->
[← 14 · One constraint, many languages](14_One_Constraint_Many_Languages.ipynb) · [Home](../../README.md) · [Equivalence of the implementations →](../../docs/EQUIVALENCE.md)